## 1. Configuration and Paths

Target catalog, schema, and volume, then derive all working paths from a single source. The landing area is split into three sub-folders with distinct roles:
- `data/` — where source JSON files land
- `_schema/` — where Auto Loader stores the inferred schema (its "schema memory")
- `_checkpoint/` — where the stream records its progress (which files have been processed)

In [0]:
dbutils.widgets.text("schema", "")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("volume", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

landing_path = f"/Volumes/{catalog}/{schema}/{volume}"

data_path = f"{landing_path}/data"
schema_path = f"{landing_path}/_schema"
checkpoint_path = f"{landing_path}/_checkpoint"

## 2. Source File Generator

A helper that generates a configurable number of JSON files, each containing a few transaction records - 2-5 per file, chosen randomly

This simulates a realistic scenario where files arrive in storage in batches over time, giving Auto Loader something to detect and ingest incrementally. The `extra_field` switch controls whether an additional `category` column is included - it is used later to demonstrate schema evolution without editing the generator mid-experiment.

In [0]:
import json
import random
from datetime import datetime, timedelta

def generate_data(num_files, start_id=0, extra_field=False):

    for i in range (start_id, start_id + num_files):
        data = []
        for j in range(random.randint(2, 5)):
            record = {
                "transaction_id": f"{i}_{j}",
                "amount": random.randint(100, 1000),
                "currency": random.choice(["USD", "EUR", "PLN"]),
                "timestamp": (datetime.now() - timedelta(minutes=random.randint(1, 10000))).isoformat() # After changing name from "date" to "timestamp" and running this notebook again name of column is still the same. It is propably because of spark caching
            }
            if extra_field:
                record["category"] = random.choice(["food", "tech", "travel"])
            data.append(record)

        file_content = "\n".join([json.dumps(record) for record in data])
        file_path = f"{data_path}/{i:05d}.json"
        dbutils.fs.put(file_path, file_content, overwrite=True)

    print(f"Generated {num_files} files from {start_id:05d} to {start_id + num_files - 1:05d} in {data_path}.")

generate_data(20, start_id=520, extra_field=True) 
display(dbutils.fs.ls(data_path))

## 3. Auto Loader Ingestion

Read the source files as a stream using Auto Loader (`cloudFiles`) and write them to a Delta table in the bronze schema.

Key design choices:
- **`readStream` + `format("cloudFiles")`** — this is Auto Loader, which incrementally detects only *new* files instead of rescanning the whole folder each run.
- **`schemaLocation`** — Auto Loader infers the schema from the files and persists it here, so it remembers the schema across runs and can detect when it changes.
- **Metadata columns** — `source_file`, `ingestion_timestamp`, and `load_date` are added to record the origin and timing of each row.
- **`outputMode("append")`** - bronze is modeled as append-only, preserving the full history of what arrived and enabling a full replay if needed. Deduplication is deferred to silver.
- **`checkpointLocation`** - the checkpoint tracks which files have been processed. This guarantees each file is ingested exactly once, so append-only ingestion produces no duplicates on re-runs, without needing a MERGE.
- **`trigger(availableNow=True)`** - process all currently available files and then stop, rather than running indefinitely. This keeps the stream controlled and cost-aware on shared infrastructure.

In [0]:
bronze_table = f"{catalog}.{schema}.transactions_stream"

df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(data_path)
)

In [0]:
from pyspark.sql import functions as F

df_bronze = (
    df_stream
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

In [0]:
query = (
    df_bronze
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")    # <- added this after evolution so this allows target delta table to accept new column
    .outputMode("append")
    .trigger(availableNow=True)       #.trigger(once=True) ("once" takes whole batch at once "availableNow" takes smaller btaches which is better for performance)
    .table(bronze_table)
)
query.awaitTermination()
print("Stream ended")

In [0]:
display(spark.sql(f"""SELECT count(*) AS num_records,\n
                  COUNT(DISTINCT source_file) AS num_files
                  FROM {bronze_table}"""))

In [0]:
display(spark.sql(f"SELECT * FROM {bronze_table} LIMIT 10"))

In [0]:
spark.catalog.refreshTable(bronze_table)
display(spark.sql(f"DESCRIBE {bronze_table}"))

In [0]:
display(spark.sql(f"""SELECT transaction_id, category, _rescued_data
                  FROM {bronze_table}
                  WHERE _rescued_data IS NOT NULL
                  LIMIT 10"""))

In [0]:
for p in query.recentProgress:
    print(f"Batch {p['batchId']} took {p['numInputRows']} rows")
    for s in p.get("sources", []):
        print(f"Sources: {s.get('metrics', {})}")


#Batch 4 took None rows
#Sources: {'numFilesOutstanding': '397', 'numBytesOutstanding': '174675', 'isBacklogComputationComplete': 'true'}
#Batch 5 took None rows
#Sources: {'numFilesOutstanding': '0', 'numBytesOutstanding': '0', 'isBacklogComputationComplete': 'true'}

# Safe reload - from zero 

In [0]:
spark.catalog.refreshTable(bronze_table)
display(spark.sql(f"SELECT COUNT(*) AS before_reload FROM {bronze_table}"))

## 7. Safe Reload — Managing Checkpoint Files

1. The target **table** (the data)
2. The **checkpoint** (which files were processed)
3. The **schema location** (the inferred schema)

After clearing all three and re-running the stream, Auto Loader treats every file as new and reloads the entire dataset. The row count after reload matched the record count in the source files exactly, confirming the reload is deterministic and safe: the same files always produce the same result, with no loss and no duplication.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")
print(f"Table deleted {bronze_table}")

dbutils.fs.rm(checkpoint_path, recurse=True)
print(f"Checkpoint deleted {checkpoint_path}")

dbutils.fs.rm(schema_path, recurse=True)
print(f"Schema location deleted {schema_path}")

print("\nReset ready - stream will start from zero.")

In [0]:
spark.catalog.refreshTable(bronze_table)
display(spark.sql(f"SELECT COUNT(*) AS after_reload FROM {bronze_table}"))